<a href="https://colab.research.google.com/github/icarof98/Studies/blob/main/Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Air temperature example
This example shows how to train a geoRF regression model on data containing spatial features. In this example, the air temperature data is used.

In [17]:
!git clone https://github.com/margotgeerts/geoRF.git

Cloning into 'geoRF'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 28 (delta 6), reused 16 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 26.73 KiB | 2.06 MiB/s, done.
Resolving deltas: 100% (6/6), done.


In [18]:
%cd geoRF

/content/geoRF/geoRF/geoRF/geoRF/geoRF


In [19]:
!pip install -r requirements.txt

  Using cached distgfs-1.1.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached geopandas-0.14.4-py3-none-any.whl.metadata (1.5 kB)
  Using cached joblib-1.4.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached kaggle-1.6.14-py3-none-any.whl
  Using cached matplotlib-3.8.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.8 kB)
  Using cached numba-0.59.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.7 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached scikit_learn-1.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached scipy-1.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached tqdm-4.66.4-py3-none-any.whl.metadata (57 k

In [20]:
import io
import requests
from urllib import request
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [44]:
# Get temperature data
# url = 'https://springernature.figshare.com/ndownloader/files/12609182'
# url_open = request.urlopen(url)
# df = pd.read_csv(io.StringIO(url_open.read().decode('utf-8')))

# Using the local file as suggested by the context
df = pd.read_csv('/content/stationDataAll.csv')

df = df[['Lat','Lon','meanT','meanP']]
df.rename(columns={"Lat":"y","Lon":"x"},inplace=True)
df

,y,x,meanT,meanP
0,36.83,7.82,18.025000,694.0
1,36.72,3.25,18.083333,672.0
2,36.72,5.07,17.366667,861.0
3,36.47,7.47,17.316667,554.0
4,36.28,6.62,15.425000,568.0
...,...,...,...,...
3071,54.65,-6.22,8.633333,961.0
3072,62.02,-6.77,6.225000,1423.0
3073,36.15,-5.35,17.441667,738.0
3074,32.63,-16.90,17.683333,612.0


In [45]:

# Convert X-Y coordinates to a projected coordinate system
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.x, df.y, crs='EPSG:4326'))

gdf = gdf.to_crs('EPSG:3857')

df['x'] = gdf.geometry.x
df['y'] = gdf.geometry.y
df


,y,x,meanT,meanP
0,4.415437e+06,8.705184e+05,18.025000,694.0
1,4.400150e+06,3.617883e+05,18.083333,672.0
2,4.400150e+06,5.643898e+05,17.366667,861.0
3,4.365487e+06,8.315566e+05,17.316667,554.0
4,4.339218e+06,7.369350e+05,15.425000,568.0
...,...,...,...,...
3071,7.294232e+06,-6.924072e+05,8.633333,961.0
3072,8.863887e+06,-7.536330e+05,6.225000,1423.0
3073,4.321281e+06,-5.955593e+05,17.441667,738.0
3074,3.846295e+06,-1.881299e+06,17.683333,612.0


In [46]:

# Split data into train and test
data_train, data_test = train_test_split(df, test_size=0.3, shuffle=True, random_state=0)

# Scale the target value
scaler = MinMaxScaler()
data_train['meanT'] = scaler.fit_transform(data_train['meanT'].values.reshape(-1,1))
data_test['meanT'] = scaler.transform(data_test['meanT'].values.reshape(-1,1))

In [47]:
target = 'meanT'
X_train = data_train.drop([target], axis=1).values
y_train = data_train[target].values

X_test = data_test.drop([target], axis=1).values
y_test = data_test[target].values

In [48]:
# Check the X-Y coordinates (column indices are required for geoRF)
X_train[:,[0,1]]

array([[  5404732.35002346, -10386319.99804493],
       [  4964404.36013148, -11186896.37998292],
       [  4162440.73969107, -11026473.86180073],
       ...,
       [  5237970.70341953, -11431888.31532075],
       [ -4184284.27482761,  16414058.91746819],
       [  7156910.262561  ,   3075757.53061815]])

In [49]:
X_train.shape

(2153, 3)

In [ ]:
from geoRF import GeoRFRegressor

ModuleNotFoundError: No module named 'distgfs'

In [51]:
!pip install distgfs
from geoRF import GeoRFRegressor

  Using cached distwq-1.2.1-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 21.6 MB/s eta 0:00:00


In [1]:
!pip install numpy==1.26.4 --force-reinstall
!pip install scikit-learn==1.4.2 --force-reinstall
georf_temp = GeoRFRegressor(n_estimators=100, max_features=None, n_jobs=-1, random_state=0)
georf_temp.fit(X_train,
               y_train,
               geo_features=[0,1], # X-Y column indices
               gens='da')  # Dual Annealing (DA) geospatial split generator

  Using cached scikit_learn-1.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached numpy-2.4.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.4.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.2 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached numpy-2.4.5-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.2 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 3.6.0
    Uninstalling threadpoolctl-3.6.0:
      Successfull

NameError: name 'GeoRFRegressor' is not defined

In [ ]:
from sklearn.metrics import mean_squared_error
print(f"train rmse: {mean_squared_error(y_train, georf_temp.predict(X_train), squared=False)}")
print(f"test rmse: {mean_squared_error(y_test, georf_temp.predict(X_test), squared=False)}")

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


train rmse: 0.014036317425465641
test rmse: 0.03631634852464978


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [ ]:
from sklearn.ensemble import RandomForestRegressor
rf_temp = RandomForestRegressor(n_estimators=100, max_features=None, n_jobs=-1, random_state=0)
rf_temp.fit(X_train, y_train)

RandomForestRegressor(max_features=None, n_jobs=-1, random_state=0)

In [ ]:
print(f"train rmse: {mean_squared_error(y_train, rf_temp.predict(X_train), squared=False)}")
print(f"test rmse: {mean_squared_error(y_test, rf_temp.predict(X_test), squared=False)}")

train rmse: 0.015041919569707926
test rmse: 0.0405312127061455


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
